# 06 — Finite-sampling analysis

The ideal-state study isolates the variational structure of QAOA from measurement uncertainty, but an experimental or shot-based simulation accesses the output distribution only through a finite number of samples. This notebook introduces that second source of error without changing the QAOA ansatz, the benchmark set or the classical optimization protocol established in the previous stage.

The analysis is deliberately performed at fixed, ideal-optimized parameters. This choice separates finite-sampling fluctuations from the additional instability that would arise if noisy expectation estimates were fed back into the classical optimizer. For each benchmark and depth, the best-energy deterministic restart of the frozen ideal protocol is reconstructed, its exact output distribution is computed, and repeated finite-shot samples are drawn from that distribution.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
    physical_feasibility_mask,
)
from oec_qaoa.finite_sampling import (
    exact_sampling_baseline,
    optimum_detection_probability,
    sample_distribution,
)
from oec_qaoa.qaoa import (
    basis_energies,
    optimize_qaoa,
    qaoa_statevector,
)

DEPTHS = [1, 2, 3, 4]
SHOTS = [128, 512, 2048, 8192]
REPETITIONS = 30
MAXITER = 200

BEST_SEEDS = {
    ("B1", 1): 101,
    ("B1", 2): 404,
    ("B1", 3): 303,
    ("B1", 4): 404,
    ("B2", 1): 303,
    ("B2", 2): 303,
    ("B2", 3): 404,
    ("B2", 4): 404,
    ("B3", 1): 202,
    ("B3", 2): 404,
    ("B3", 3): 303,
    ("B3", 4): 404,
}

## Sampling protocol

For an ideal QAOA state with computational-basis probabilities (p_z), the exact expectation is

[
langle H_Cangle=sum_z p_z C(z).
]

A finite-shot experiment replaces this quantity by the empirical estimator

[
widehat{langle H_Cangle}
=
rac{1}{N_{mathrm{shots}}}
sum_{s=1}^{N_{mathrm{shots}}}
C(z_s),
]

with (z_s) sampled from the exact output distribution. The same samples provide empirical estimates of the optimal-state and feasible-state probabilities.

The shot budgets considered here are

[
N_{mathrm{shots}}in{128,512,2048,8192},
]

and each configuration is repeated thirty times with deterministic sampling seeds. In addition to the sampled probabilities, the analysis records whether at least one exact optimum is observed. If the ideal optimal probability is (P_{mathrm{opt}}), the corresponding analytic detection probability is

[
P_{mathrm{detect}}
=
1-(1-P_{mathrm{opt}})^{N_{mathrm{shots}}}.
]

This quantity is useful when (P_{mathrm{opt}}) is small, because an accurate estimate of the expectation value does not necessarily imply that the optimum is ever sampled.

In [ ]:
def cost_scale(model) -> float:
    coefficients = [
        abs(float(value))
        for value in model.linear
        if abs(float(value)) > 1e-12
    ]
    coefficients.extend(
        abs(float(value))
        for value in model.quadratic.values()
        if abs(float(value)) > 1e-12
    )
    return max(coefficients) if coefficients else 1.0


records = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    feasible_mask = physical_feasibility_mask(instance, retained, model)

    energies = basis_energies(model)
    optimum = float(energies.min())
    optimal_mask = np.isclose(energies, optimum, atol=1e-9)
    phase_energies = energies / cost_scale(model)

    for depth in DEPTHS:
        seed = BEST_SEEDS[(benchmark.benchmark_id, depth)]
        result = optimize_qaoa(
            model,
            depth=depth,
            seed=seed,
            feasible_mask=feasible_mask,
            optimizer="COBYLA",
            maxiter=MAXITER,
            normalize_cost=True,
        )

        state = qaoa_statevector(
            phase_energies,
            result.parameters[:depth],
            result.parameters[depth:],
        )
        probabilities = np.abs(state) ** 2
        baseline = exact_sampling_baseline(
            probabilities,
            energies,
            optimal_mask,
            feasible_mask,
        )

        for shots in SHOTS:
            for repetition in range(REPETITIONS):
                sample_seed = (
                    1_000_000
                    + 10_000 * int(benchmark.benchmark_id[1:])
                    + 1_000 * depth
                    + 10 * SHOTS.index(shots)
                    + repetition
                )
                sampled = sample_distribution(
                    probabilities,
                    energies,
                    optimal_mask,
                    feasible_mask,
                    shots=shots,
                    seed=sample_seed,
                )

                records.append(
                    {
                        "benchmark_id": benchmark.benchmark_id,
                        "depth": depth,
                        "shots": shots,
                        "repetition": repetition,
                        "sampled_expected_energy": (
                            sampled.sampled_expected_energy
                        ),
                        "sampled_optimal_probability": (
                            sampled.sampled_optimal_probability
                        ),
                        "sampled_feasible_probability": (
                            sampled.sampled_feasible_probability
                        ),
                        "best_sampled_energy": sampled.best_sampled_energy,
                        "optimum_observed": sampled.optimum_observed,
                        "ideal_expected_energy": baseline[
                            "exact_expected_energy"
                        ],
                        "ideal_optimal_probability": baseline[
                            "exact_optimal_probability"
                        ],
                        "ideal_feasible_probability": baseline[
                            "exact_feasible_probability"
                        ],
                        "analytic_optimum_detection_probability": (
                            optimum_detection_probability(
                                baseline["exact_optimal_probability"],
                                shots,
                            )
                        ),
                    }
                )

runs = pd.DataFrame(records)
runs.head()

## Aggregated sampling error

The repeated samples are summarized through the mean and standard deviation of the sampled energy, the empirical optimal and feasible probabilities, and the frequency with which at least one optimum is observed. The exact ideal-state values are retained beside these estimates so that the effect of the shot budget can be read directly.

This stage should not be interpreted as a noisy-hardware experiment. The quantum state itself remains ideal, and the only stochasticity introduced here is the finite number of computational-basis measurements.

In [ ]:
summary = (
    runs.groupby(["benchmark_id", "depth", "shots"])
    .agg(
        sampled_expected_energy_mean=(
            "sampled_expected_energy",
            "mean",
        ),
        sampled_expected_energy_std=(
            "sampled_expected_energy",
            "std",
        ),
        sampled_optimal_probability_mean=(
            "sampled_optimal_probability",
            "mean",
        ),
        sampled_feasible_probability_mean=(
            "sampled_feasible_probability",
            "mean",
        ),
        optimum_observation_rate=("optimum_observed", "mean"),
        best_sampled_energy_mean=("best_sampled_energy", "mean"),
        ideal_expected_energy=("ideal_expected_energy", "first"),
        ideal_optimal_probability=(
            "ideal_optimal_probability",
            "first",
        ),
        ideal_feasible_probability=(
            "ideal_feasible_probability",
            "first",
        ),
        analytic_optimum_detection_probability=(
            "analytic_optimum_detection_probability",
            "first",
        ),
    )
    .reset_index()
)

summary

## Interpretation

The finite-sampling stage answers a different question from the ideal-QAOA benchmark. The ideal probabilities characterize how much weight the optimized variational state places on useful basis states; the shot experiment determines how reliably those probabilities can be observed at a finite measurement budget.

In particular, when (P_{mathrm{opt}}) is of order (10^{-4}) or smaller, several thousand shots may still provide a reasonable estimate of the expected energy while frequently failing to observe the exact optimum. This distinction is important for the OEC setting because a sampled feasible allocation can remain useful even when the globally optimal configuration is rare.

The following stage will decide whether adding an explicit device-noise model or a hardware-oriented compilation study contributes additional information beyond this finite-sampling effect. Such an extension will be included only if it changes the scientific interpretation rather than serving as a purely demonstrative hardware example.